# MLP Curve Fitting

AI & ML Foundations: Neural networks and backpropagation · *Mathematics for AI and Machine Learning* by Fuheng Wu

Train a one-hidden-layer network in the browser and watch wider layers fit a curve better.

Companion code for the [interactive demo](https://math4ai.org/companion/ml/mlp-fit/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(7)

# The demo's setup: 40 noisy samples of sin(2x) + 0.3x on [-2, 2], one tanh hidden layer, Adam
x = torch.linspace(-2, 2, 40).unsqueeze(1)
y = torch.sin(2 * x) + 0.3 * x + 0.1 * torch.randn_like(x)
x_test = torch.linspace(-2, 2, 400).unsqueeze(1)
y_test = torch.sin(2 * x_test) + 0.3 * x_test              # the noise-free curve


def train(hidden, steps=1500, lr=0.02):
    net = torch.nn.Sequential(torch.nn.Linear(1, hidden), torch.nn.Tanh(), torch.nn.Linear(hidden, 1))
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        loss = ((net(x) - y) ** 2).mean()
        loss.backward()
        opt.step()
    with torch.no_grad():
        test = ((net(x_test) - y_test) ** 2).mean().item()
    return loss.item(), test, sum(p.numel() for p in net.parameters())


# ── More hidden units fit better, until the noise is the limit ──────────────
print(" hidden H   parameters   train MSE   error vs true curve")
for H in (1, 2, 4, 8, 32):
    tr, te, n = train(H)
    print(f"{H:8d}   {n:10d}   {tr:9.4f}   {te:9.4f}")
print("\nThe noise variance is 0.01: past a few units the train MSE sits near it, and a much")
print("wider net starts to follow the noise (its error against the true curve creeps back up).")

# ── What one hidden unit contributes: a single smooth step ──────────────────
net = torch.nn.Sequential(torch.nn.Linear(1, 1), torch.nn.Tanh(), torch.nn.Linear(1, 1))
with torch.no_grad():
    net[0].weight.fill_(3.0); net[0].bias.fill_(-1.5); net[2].weight.fill_(0.8)
print("one unit, v tanh(3x - 1.5): values at x = -1, 0.5, 2:",
      [round(v, 3) for v in net(torch.tensor([[-1.0], [0.5], [2.0]])).squeeze().tolist()])

 hidden H   parameters   train MSE   error vs true curve


       1            4      0.1924      0.1624


       2            7      0.0136      0.0055


       4           13      0.0077      0.0018


       8           25      0.0075      0.0019


      32           97      0.0070      0.0023

The noise variance is 0.01: past a few units the train MSE sits near it, and a much
wider net starts to follow the noise (its error against the true curve creeps back up).
one unit, v tanh(3x - 1.5): values at x = -1, 0.5, 2: [-0.992, -0.192, 0.608]
